In [ ]:
import pandas as pd

df = pd.read_csv('../../data/vacancies.csv')

In [ ]:
df.columns

In [ ]:
df['employer_name'].nunique()

In [10]:
values = df['employer_name'].value_counts().head(1000).index.tolist()
pd.Series(values).to_csv('top_firms.csv', index=False, header=False)

In [ ]:
df[['name', 'description', 'employer_name', 'raw_skills']].value_counts()

In [ ]:
result = df[df['employer_name'].isna() & df['name'].notna()]

In [ ]:
result['data_source'].value_counts()

In [ ]:
imp_cols = ['name', 'employer_name']

df.dropna(subset=imp_cols)[['employer_name', 'name', 'description', 'raw_skills']].drop_duplicates().to_csv('../../data/vacancies_clean.csv')

In [ ]:
df = pd.read_csv('../../data/vac_labeled.csv')

In [ ]:
top_ind1 = df.nlargest(2000, 'ai_relatedness')
mask_empty = df['raw_skills'] == '[]'
top_ind2 = df[mask_empty].nlargest(500, 'ai_potential')
result = pd.concat([top_ind1, top_ind2], ignore_index=True)[['Unnamed: 0', 'name', 'description', 'raw_skills']].rename(columns={'Unnamed: 0': 'id'})
result.to_csv("../../data/train1.csv")

In [ ]:
df = df.sort_values(by='ai_relatedness')
df.head(500)

In [ ]:
df[df['ai_relatedness'] > 0.1]['employer_name'].value_counts()

In [ ]:
import numpy as np
top_1000 = df['employer_name'].value_counts().head(1011).index
mask = df['employer_name'].isin(top_1000) & (df['raw_skills'] != '[]')
mask2 = df['description'].notnull()
filtered = df[mask & mask2].copy()
idx_best = filtered.groupby('employer_name')['ai_relatedness'].idxmin()
result = filtered.loc[idx_best].reset_index(drop=True)
top_ind2 = result.rename(columns={'Unnamed: 0.1': 'Unnamed: 0', 'Unnamed: 0': 'id'})
top_ind2 = top_ind2.drop(columns=['employer_name'])
top_ind2['score'] = np.zeros(len(top_ind2))
top_ind2

In [ ]:
train = pd.read_csv('../../data/train1.csv')

In [ ]:
new_train = pd.concat([train, top_ind2], ignore_index=True)
new_train.to_csv('../../data/train_bigger.csv')

In [ ]:
df1 = pd.read_csv('../../data/vac_labeled.csv')

In [ ]:
merged = new_train.merge(df1, left_on='id', right_on='Unnamed: 0', how='left')
merged

In [ ]:
for i in range(11):
    print(sum(i / 10 <= merged['score']) - sum((i+1) / 10 <= merged['score']))

In [ ]:
df_sorted = merged.sort_values('score').reset_index(drop=True)
n_parts = 25
part_size = len(df_sorted) // n_parts

parts = []
for i in range(n_parts):
    start = i * part_size
    end = (i + 1) * part_size if i < n_parts - 1 else len(df_sorted)
    parts.append(df_sorted.iloc[start:end])
samples = []
for i, part in enumerate(parts):
    n = min(20, len(part))
    sample = part.sample(n=n, random_state=42)
    samples.append(sample)
final_df = pd.concat(samples).reset_index(drop=True)
final_df

In [20]:
final_df.to_csv('golden_man.csv')

In [ ]:
df_use = df[['Unnamed: 0', 'name', 'description', 'raw_skills', 'ai_relatedness']].rename(columns={'Unnamed: 0': 'id'})
df1 = pd.read_csv('../../golden_scores.csv')
df2 = pd.read_csv('../../scores_man.csv')
merged = df_use.merge(df1, on='id', how='right')
merged

In [5]:
train = merged[~merged['id'].isin(df2['id'])]
test = merged.merge(df2.rename(columns={'score': 'true_score'}), on='id', how='right')

In [ ]:
test.to_csv('../../data/test_manual.csv')
train.to_csv('../../data/train_manual.csv')